# HAM10000 Skin Lesion Classification Pipeline

**Workflow:** GitHub → Kaggle GPU

> Run the manual setup cells below FIRST (clone repo, cd, pip install), then run the remaining cells.

> **Not medical advice** — research/education only.

## A — Manual setup (YOU run these first)

Uncomment and set your GitHub URL, then run:

```python
# !git clone https://github.com/YOUR_USERNAME/SkinCancerDetection.git
# %cd SkinCancerDetection
# !pip install -q -r requirements.txt
```

## B — Environment bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Ensure repo root is on sys.path (after %cd SkinCancerDetection)
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Kaggle environment: use Kaggle dataset paths from config
os.environ["SKIN_CANCER_ENV"] = "kaggle"

print("Project root:", PROJECT_ROOT)
print("SKIN_CANCER_ENV:", os.environ["SKIN_CANCER_ENV"])

In [ ]:
# Optional: override Kaggle dataset slug if your attached dataset path differs
KAGGLE_DATA_SLUG = "skin-cancer-mnist-ham10000"

PATH_OVERRIDES = {
    "data.kaggle_metadata_csv": f"/kaggle/input/{KAGGLE_DATA_SLUG}/HAM10000_metadata.csv",
    "data.kaggle_images_dirs": [
        f"/kaggle/input/{KAGGLE_DATA_SLUG}/HAM10000_images_part_1",
        f"/kaggle/input/{KAGGLE_DATA_SLUG}/HAM10000_images_part_2",
    ],
}

## C — Imports (public APIs only)

In [ ]:
from src.config import load_config
from src.train import train_model
from src.evaluate import evaluate_model
from src.visualize import run_gradcam, predict_image, plot_metrics

config = load_config(overrides=PATH_OVERRIDES)
print("Config env:", config["env"])
print("Metadata:", config["data"]["metadata_csv"])

## D — Training

Full training uses all HAM10000 images. For a quick GPU smoke test, set `debug_max_samples` and fewer epochs.

In [ ]:
# Full training run
TRAIN_OVERRIDES = {
    **PATH_OVERRIDES,
    "training.epochs": 30,
    "training.batch_size": 32,
    "model.name": "efficientnet_v2_s",
    "data.num_workers": 2,
}

# Uncomment for smoke test only:
# TRAIN_OVERRIDES["data.debug_max_samples"] = 500
# TRAIN_OVERRIDES["training.epochs"] = 2

train_result = train_model(overrides=TRAIN_OVERRIDES)
train_result

## E — Evaluation

In [ ]:
test_metrics = evaluate_model(overrides=PATH_OVERRIDES, split="test")
print(f"Test accuracy:       {test_metrics['accuracy']:.4f}")
print(f"Balanced accuracy:   {test_metrics['balanced_accuracy']:.4f}")
print(f"Macro F1:            {test_metrics['macro_f1']:.4f}")

In [ ]:
val_metrics = evaluate_model(overrides=PATH_OVERRIDES, split="val")
val_metrics

## F — Metrics plotting

In [ ]:
from IPython.display import Image, display

curves_path = plot_metrics(overrides=PATH_OVERRIDES)
display(Image(filename=curves_path))

In [ ]:
cm_path = Path(config["paths"]["figures_dir"]) / "confusion_matrix_test.png"
if cm_path.is_file():
    display(Image(filename=str(cm_path)))
else:
    print("Run evaluate_model(split='test') first.")

## G — Grad-CAM visualization

In [ ]:
# Example: set an image_id from the test set or full path to a .jpg
EXAMPLE_IMAGE_ID = "ISIC_0024306"  # change to a valid image_id in your dataset

gradcam_result = run_gradcam(
    EXAMPLE_IMAGE_ID,
    overrides=PATH_OVERRIDES,
)
print("Predicted:", gradcam_result["predicted_class"], "| confidence:", gradcam_result["confidence"])
print("Saved:", gradcam_result.get("gradcam_save_path"))

In [ ]:
if gradcam_result.get("gradcam_save_path"):
    display(Image(filename=gradcam_result["gradcam_save_path"]))

## H — Inference demo

In [ ]:
# Use full path if needed
pred = predict_image(
    EXAMPLE_IMAGE_ID,
    overrides=PATH_OVERRIDES,
)
pred

## I — Per-class metrics summary

In [ ]:
import json
from src.constants import CLASS_NAMES

report_path = Path(config["paths"]["reports_dir"]) / "metrics_test.json"
with open(report_path) as f:
    report = json.load(f)

for name in CLASS_NAMES:
    pc = report["per_class"][name]
    print(f"{name:6s} | P={pc['precision']:.3f} R={pc['recall']:.3f} F1={pc['f1']:.3f} n={pc['support']}")